In [ ]:
import math
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

import torch
from torch import nn
import torch.nn.functional as F
import torch.distributed as dist

from kernel import act_quant, weight_dequant, fp8_gemm

In [ ]:
world_size = 1
rank = 0
block_size = 128
# 控制矩阵乘法（GEMM - General Matrix Multiply）的计算方式
gemm_impl: Literal["bf16", "fp8"] = "bf16"
# 控制多头潜在注意力（MLA - Multi-Head Latent Attention）的实现方式
attn_impl: Literal["naive", "absorb"] = "absorb"

In [ ]:
# 模型参数
@dataclass
class ModelArgs:
    max_batch_size: int = 8
    max_seq_len: int = 4096 * 4
    dtype: Literal["bf16", "fp8"] = "bf16"
    vocab_size: int = 102400
    dim: int = 4096
    inter_dim: int = 11944
    moe_inter_dim: int = 1408
    n_layers: int = 27
    n_dense_layers: int = 1
    n_heads: int = 16
    # moe
    n_routed_experts: int = 64
    n_shared_experts: int = 2
    n_activated_experts: int = 6
    n_expert_groups: int = 1
    n_limited_groups: int = 1
    score_func: Literal["softmax", "sigmoid"] = "softmax"
    route_scale: float = 1.0
    # mla
    q_lora_rank: int = 0
    kv_lora_rank: int = 512
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128
    # yarn
    original_seq_len: int = 4096
    rope_theta: float = 10000.0
    rope_factor: float = 40
    beta_fast: int = 32
    beta_slow: int = 1
    mscale: float = 1.0

In [ ]:
class ParallelEmbedding(nn.Module):
    """支持并行的Embedding层"""
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        assert vocab_size % world_size == 0, f"Vocabulary size must be divisible by world size (world_size={world_size})"
        self.part_vocab_size = (vocab_size // world_size)
        self.vocab_start_idx = rank * self.part_vocab_size
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if world_size > 1:
            # 取出当前进程负责范围内的Embedding
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)
            x = x - self.vocab_start_idx
            x[mask] = 0
        y = F.embedding(x, self.weight)
        if world_size > 1:
            # 只保留当前进程负责的部分
            y[mask] = 0
            # 将每个进程的张量进行求和，并将结果广播给所有进程
            dist.all_reduce(y)
        return y

In [ ]:
def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
    """自定义的线性运行,用于支持量化的相关操作"""
    # 标准精度运算
    # > 1：表示权重是标准精度（如float32等）
    # == 1：表示权重是量化权重（如int8等）
    if weight.element_size() > 1:
        return F.linear(x, weight, bias)
    # bfloat16反量化运算
    elif gemm_impl == "bf16":
        weight = weight_dequant(weight, weight.scale)
        return F.linear(x, weight, bias)
    # FP8量化运算
    else:
        # 量化时，同一个Block内的变量共享同一个缩放因子scale
        x, scale = act_quant(x, block_size)
        # 使用FP8量化的矩阵乘法
        y = fp8_gemm(x, scale, weight, weight.scale)
        # 偏置项不量化
        if bias is not None:
            y += bias
        return y


In [ ]:
class Linear(nn.Module):
    """自定义线性层, 用于支持量化操作"""
    pass